# Universal AC-MOT — Colab T4 session (Part A: caches only · Part B: official benchmark)
Part A builds detection caches and image-cue caches (inference only — **no metrics are computed here**).
Part B (official T4 timing) is run only after the final V5/V4 decision is frozen; it times the code at the commit it prints.
Uses a fresh clone; does not touch `/content/Universal-ACMOT`.

In [ ]:
# A1. GPU + Drive
import subprocess, torch, os, shutil, json, hashlib
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'needs a T4 runtime'
from google.colab import drive; drive.mount('/content/drive')
D = '/content/drive/MyDrive'

In [ ]:
# A2. Fresh clone of the pushed branch
%cd /content
!rm -rf /content/Universal-ACMOT-v5
!git clone -q --branch universal-adapters-v1 https://github.com/AhmedCode110/AC-MOT.git /content/Universal-ACMOT-v5
%cd /content/Universal-ACMOT-v5
!git log --oneline -1
!pip -q install ultralytics==8.3.200 lap==0.5.12

In [ ]:
# A3. Weights (+ sha256)
os.makedirs('weights', exist_ok=True)
for w in ['yolov8n.pt', 'rtdetr-l.pt']:
    if os.path.exists(f'/content/Universal-ACMOT/{w}'): shutil.copy(f'/content/Universal-ACMOT/{w}', w)
!test -f yolov8n.pt || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt
!test -f rtdetr-l.pt || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/rtdetr-l.pt
!test -f weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth || wget -q -P weights https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
EXPECT = {'yolov8n.pt':'f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36',
          'rtdetr-l.pt':'6de60b10d4bc566f00cda0f5b4d64afe4b66d48dc9695d2171effb7859d8e73f',
          'weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth':'dd69338a24b8d7381807e247652bdc356325bcbaf1cd3e092e00e0a1a58706bf'}
for p, h in EXPECT.items(): print(p, hashlib.sha256(open(p,'rb').read()).hexdigest() == h)

In [ ]:
# A4. Datasets -> local disk
TRAIN_SRC = f'{D}/AC-MOT-shared/AC-MOT-data/VisDrone2019-MOT-train'
VAL_SRC   = f'{D}/AC-MOT-shared/AC-MOT-data/VisDrone2019-MOT-val'
TD_SRC    = f'{D}/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev'
UA_SRC    = f'{D}/AC-MOT-shared/UAVDT_EXTERNAL_GENERALIZATION/data/extracted/UAV-benchmark-M/UAV-benchmark-M'
UA_TEST = ['M0203','M0205','M0208','M0209','M0403','M0601','M0602','M0606','M0701','M0801',
           'M0802','M1001','M1004','M1007','M1009','M1101','M1301','M1302','M1303','M1401']
!test -d /content/VisDrone2019-MOT-train || cp -r "$TRAIN_SRC" /content/VisDrone2019-MOT-train
!test -d /content/VisDrone2019-MOT-val || cp -r "$VAL_SRC" /content/VisDrone2019-MOT-val
!test -d /content/VisDrone2019-MOT-test-dev || cp -r "$TD_SRC" /content/VisDrone2019-MOT-test-dev
os.makedirs('/content/uavdt_view/sequences', exist_ok=True)
for s in UA_TEST:
    if not os.path.isdir(f'/content/uavdt_view/sequences/{s}'):
        shutil.copytree(f'{UA_SRC}/{s}', f'/content/uavdt_view/sequences/{s}')
for p in ['/content/VisDrone2019-MOT-train','/content/VisDrone2019-MOT-val','/content/VisDrone2019-MOT-test-dev','/content/uavdt_view']:
    print(p, len(os.listdir(p + '/sequences')), 'sequences')

In [ ]:
# A5. Detection caches (inference only). Output -> Drive/Universal-ACMOT-Results/t4_caches/<set>/<detector>
OUT = f'{D}/Universal-ACMOT-Results/t4_caches'
FR = 'weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth'
jobs = [
  # V5 development set (never used before): YOLOv8n + RT-DETR-L, all levels
  ('yolov8n.pt',  '/content/VisDrone2019-MOT-train', 'train',  'yolov8',     '0.45', '640 736 832'),
  ('rtdetr-l.pt', '/content/VisDrone2019-MOT-train', 'train',  'rtdetr',     '0.45', '640 736 832'),
  # Unseen detector (transfer only): Faster R-CNN on val / test-dev / UAVDT
  (FR, '/content/VisDrone2019-MOT-val',      'val',     'fasterrcnn', '0.45', '640 736 832'),
  (FR, '/content/VisDrone2019-MOT-test-dev', 'testdev', 'fasterrcnn', '0.45', '640 736 832'),
  (FR, '/content/uavdt_view',                'uavdt',   'fasterrcnn', '0.45', '640 736 832'),
  # NMS audit (torchvision-native 0.5)
  (FR, '/content/VisDrone2019-MOT-val',      'val_nms050', 'fasterrcnn', '0.5', '736'),
]
for w, ds, tag, det, nms, res in jobs:
    out = f'{OUT}/{tag}/{det}'
    !PYTHONPATH=. python -u tools/cache_detections.py --weights $w --dataset "$ds" --output-dir "$out" --nms $nms --resolutions $res 2>&1 | grep -v -i warn | tail -2

In [ ]:
# A6. Image-cue caches (edges, brightness, blur, global motion) — CPU
for ds, tag in [('/content/VisDrone2019-MOT-train','train'), ('/content/VisDrone2019-MOT-test-dev','testdev'), ('/content/uavdt_view','uavdt')]:
    !PYTHONPATH=. python tools/visual_cues.py "$ds" "$OUT/$tag"
!ls -R "$OUT" | head -40

## Part B — OFFICIAL T4 benchmark (run ONLY after the final freeze is announced)
Checks out the frozen tag, verifies the lock hashes, and times detector / tracker / adaptive layer.

In [ ]:
# B1. (after freeze) set TAG to the announced final tag, then run
TAG = 'REPLACE_WITH_FINAL_TAG'
%cd /content/Universal-ACMOT-v5
!git fetch -q --tags && git checkout -q $TAG && git log --oneline -1
!PYTHONPATH=. python tools/t4_benchmark.py --dataset /content/VisDrone2019-MOT-val --sequence uav0000137_00458_v \
   --weights yolov8n.pt rtdetr-l.pt weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth \
   --levels 640 736 832 --trackers bytetrack botsort --frames 300 --warmup 20 \
   --out "$D/Universal-ACMOT-Results/t4_benchmark_$TAG.json" 2>&1 | grep -v -i warn
!pip freeze | grep -i -E "torch|ultralytics|numpy|opencv|lap" > "$D/Universal-ACMOT-Results/t4_packages_$TAG.txt"